# ChangeOfBase (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch05_numerical_algorithms/ChangeOfBase_cpp.ipynb)

Chapter 5 — Numerical algorithms. Runnable notebook of `ChangeOfBase`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/basic-algorithms/05-numerical-algorithms/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch05_numerical_algorithms/ChangeOfBase_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch05_numerical_algorithms/ChangeOfBase_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch05_numerical_algorithms_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Indices.** In the pseudo-code arrays start from 1, here from 0: the element $a[i]$ of the pseudo-code is `a[i-1]` in the code. The printed traces use the indices of the code (as in the C++ examples of the lecture notes), so they are those of the pseudo-code with every index decreased by 1. The function names are those of the pseudo-code, as in the lecture notes (`FindMaximum`).

## ChangeOfBase

**Input:** a natural number $n \ge 1$ (in base 10) and a base $b \ge 2$. **Output:** the array $d[1..nd]$ of the digits of $n$ in base $b$ and the number of digits $nd$.

One division by $b$ extracts the least significant digit ($r = n \bmod b$) and leaves as quotient the number written with the other digits: the division is repeated until the quotient is 0. The digits come out from the least significant to the most significant, so in the array `d` the digit of weight $b^0$ is in `d[0]` (in $d[1]$ in the pseudo-code).

The number of digits $nd = \lfloor \log_b n \rfloor + 1$ is computed here with integer multiplications only, without floating point logarithms: in double precision `log(1000)/log(10)` evaluates to $2.9999999999999996$ and the formula would give 3 digits instead of 4.

In [1]:
%%writefile change_of_base.cpp
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <string>
#include <vector>
using namespace std;

// Helper function to print the trace: the first n entries as "[1, 1, 1]"
string Str(int n, const int *a)
{
    string s = "[";
    for (int i = 0; i < n; i++) s += (i ? ", " : "") + to_string(a[i]);
    return s + "]";
}

string Join(const vector<int> &v)       // the entries separated by spaces
{
    string s;
    for (size_t i = 0; i < v.size(); i++) s += (i ? " " : "") + to_string(v[i]);
    return s;
}

// Returns the digits of n in base b: d[0] is the least significant digit, d.size() is nd
vector<int> ChangeOfBase(int n, int b, bool trace = false)
{
    int nd = 1;                         // nd = ⌊log_b n⌋ + 1, with integer operations only
    for (long long p = b; p <= n; p *= b)
        nd = nd + 1;
    vector<int> d(nd);                  // array of the digits
    int k = 0;                          // k = 1 in the pseudo-code
    while (n > 0)
    {
        int q = n / b;                  // quotient
        int r = n % b;                  // remainder (digit in base b)
        d[k] = r;
        n = q;
        k = k + 1;
        if (trace)
            printf("%10d %5d %3d %5d %3d  %s\n", k, q, r, n, k, Str(k, d.data()).c_str());
    }
    return d;
}

string DigitString(const vector<int> &d)    // the digits from the most to the least significant
{
    string s;
    for (int i = d.size() - 1; i >= 0; i--) s += "0123456789ABCDEF"[d[i]];
    return s;
}

void Run(int n, int b)
{
    printf("Conversion: n = %d, b = %d\n", n, b);
    printf("%10s %5s %3s %5s %3s  %s\n", "iteration", "q", "r", "n", "nd", "digits");
    vector<int> d = ChangeOfBase(n, b, true);
    printf("Return: nd = %d, d = %s\n", (int)d.size(), Str(d.size(), d.data()).c_str());
    printf("%d = (%s)_%d\n\n", n, DigitString(d).c_str(), b);
}

int main(int argc, char *argv[])
{
    if (argc > 2)   // “Your turn”: n and the base b from the command line
    {
        Run(atoi(argv[1]), atoi(argv[2]));
        return 0;
    }
    Run(7, 2);
    Run(8, 2);
    Run(25, 3);
    Run(100, 8);
    Run(1000, 10);    // 4 digits: the case where the floating point log(1000)/log(10) fails
    return 0;
}

Writing change_of_base.cpp


### Execution on the examples of the chapter

In [2]:
!g++ -std=c++17 -O2 -o change_of_base change_of_base.cpp && ./change_of_base

Conversion: n = 7, b = 2
 iteration     q   r     n  nd  digits
         1     3   1     3   1  [1]
         2     1   1     1   2  [1, 1]
         3     0   1     0   3  [1, 1, 1]
Return: nd = 3, d = [1, 1, 1]
7 = (111)_2

Conversion: n = 8, b = 2
 iteration     q   r     n  nd  digits
         1     4   0     4   1  [0]
         2     2   0     2   2  [0, 0]
         3     1   0     1   3  [0, 0, 0]
         4     0   1     0   4  [0, 0, 0, 1]
Return: nd = 4, d = [0, 0, 0, 1]
8 = (1000)_2

Conversion: n = 25, b = 3
 iteration     q   r     n  nd  digits
         1     8   1     8   1  [1]
         2     2   2     2   2  [1, 2]
         3     0   2     0   3  [1, 2, 2]
Return: nd = 3, d = [1, 2, 2]
25 = (221)_3

Conversion: n = 100, b = 8
 iteration     q   r     n  nd  digits
         1    12   4    12   1  [4]
         2     1   4     1   2  [4, 4]
         3     0   1     0   3  [4, 4, 1]
Return: nd = 3, d = [4, 4, 1]
100 = (144)_8

Conversion: n = 1000, b = 10
 iteration     q   r

### Try it yourself

Change the numbers below and run the cell again. The numbers are $n$ and the base $b$ (up to 16: the digits above 9 are written A, B, …).

In [3]:
!./change_of_base 255 16

Conversion: n = 255, b = 16
 iteration     q   r     n  nd  digits
         1    15  15    15   1  [15]
         2     0  15     0   2  [15, 15]
Return: nd = 2, d = [15, 15]
255 = (FF)_16



---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).